# Linear Probe: Dev 12-Lead Ablation

Evaluate the dev-preset checkpoint with the highest pooled eRank from the 12-lead run. PTB-XL is kept in the model's training geometry: all 12 leads and 2500 samples per record.

In [1]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
if not (ROOT / "src" / "encoder.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "src" / "encoder.py").is_file():
    ROOT = Path("/home/aimakeradmin/shady/TS-JEPA")
sys.path.insert(0, str(ROOT))

from collapse_investigation.analysis_utils import (
    best_available_checkpoint,
    build_ours_encoder,
    load_encoder_weights,
    pooled_features,
    prepare_signal,
)
from src.data.ptbxl_dataset import PTBXLDataset

RUN_NAME = "dev_12lead"
RUN_DIR = ROOT / "checkpoints" / RUN_NAME
METRICS_CSV = RUN_DIR / "metrics.csv"
PTBXL_DATA_DIR = ROOT / "data" / "ptbxl_500hz_2500_raw"
FIG_DIR = ROOT / "collapse_investigation" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)
print("ROOT:", ROOT)
print("device:", device)
print("PTBXL_DATA_DIR:", PTBXL_DATA_DIR)

ROOT: /home/aimakeradmin/shady/TS-JEPA
device: cuda
PTBXL_DATA_DIR: /home/aimakeradmin/shady/TS-JEPA/data/ptbxl_500hz_2500_raw


In [ ]:
BEST_POOL_EPOCH, CHECKPOINT, BEST_POOL_ERANK = best_available_checkpoint(RUN_DIR, METRICS_CSV, "erank_ctx_pool")
print("Best available pooled-eRank epoch:", BEST_POOL_EPOCH)
print("Best available pooled eRank:", f"{BEST_POOL_ERANK:.4f}")
print("Checkpoint:", CHECKPOINT)

encoder, tokenizer, enc_cfg = build_ours_encoder(RUN_NAME, ROOT, use_flash=True)
load_encoder_weights(encoder, CHECKPOINT) 
encoder = encoder.to(device).eval() 
for param in encoder.parameters():
    param.requires_grad_(False)

print("encoder config:", enc_cfg)
print("expected wave shape:", (enc_cfg.num_leads, enc_cfg.num_patches * enc_cfg.patch_size))

Best available pooled-eRank epoch: 32
Best available pooled eRank: 22.9942
Checkpoint: /home/aimakeradmin/shady/TS-JEPA/checkpoints/dev_12lead/checkpoint_epoch_33.pt
encoder config: EncoderConfig(num_leads=12, patch_size=50, num_patches=50, embed_dim=384, depth=6, num_heads=8, mlp_ratio=4.0, dropout=0.0, drop_path=0.1, use_flash=True, qkv_bias=True)
expected wave shape: (12, 2500)


In [3]:
trainset = PTBXLDataset(PTBXL_DATA_DIR, split="train", return_labels=True)
valset = PTBXLDataset(PTBXL_DATA_DIR, split="val", return_labels=True)
testset = PTBXLDataset(PTBXL_DATA_DIR, split="test", return_labels=True)
print("train/val/test:", len(trainset), len(valset), len(testset))

xb = torch.stack([trainset[i][0] for i in range(2)]).to(device)
yb = prepare_signal(xb, RUN_NAME)
patches = tokenizer.patchify(yb)
with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device.type == "cuda" and enc_cfg.use_flash)):
    zb = encoder.forward_all(patches).mean(dim=1)
print("raw batch:", tuple(xb.shape))
print("prepared batch:", tuple(yb.shape))
print("patches:", tuple(patches.shape))
print("pooled features:", tuple(zb.shape))

train/val/test: 17111 2156 2163
raw batch: (2, 12, 2500)
prepared batch: (2, 12, 2500)
patches: (2, 12, 50, 50)
pooled features: (2, 384)


In [4]:
def effective_rank(x: torch.Tensor, eps: float = 1e-12) -> float:
    x = x.float()
    x = x - x.mean(dim=0, keepdim=True)
    s = torch.linalg.svdvals(x)
    p = s / s.sum().clamp_min(eps)
    entropy = -(p * (p + eps).log()).sum()
    return float(entropy.exp().cpu())


def extract_features(dataset, batch_size: int = 256):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    features, labels = [], []
    encoder.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda" and enc_cfg.use_flash)):
                waves = prepare_signal(x, RUN_NAME)
                z = pooled_features(encoder, tokenizer, waves)
            features.append(z.float().cpu())
            labels.append(y.float().cpu())
    return torch.cat(features), torch.cat(labels)

train_features, train_labels = extract_features(trainset)
val_features, val_labels = extract_features(valset)
test_features, test_labels = extract_features(testset)
print("train features:", train_features.shape, "eRank", effective_rank(train_features))
print("val features:", val_features.shape, "eRank", effective_rank(val_features))
print("test features:", test_features.shape, "eRank", effective_rank(test_features))

train features: torch.Size([17111, 384]) eRank 107.20845794677734
val features: torch.Size([2156, 384]) eRank 104.79476165771484
test features: torch.Size([2163, 384]) eRank 105.20476531982422


In [5]:
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score

NUM_CLASSES = train_labels.shape[1]
probe_train = TensorDataset(train_features, train_labels)
probe_val = TensorDataset(val_features, val_labels)
probe_test = TensorDataset(test_features, test_labels)
probe_train_loader = DataLoader(probe_train, batch_size=512, shuffle=True)
probe_val_loader = DataLoader(probe_val, batch_size=512, shuffle=False)
probe_test_loader = DataLoader(probe_test, batch_size=512, shuffle=False)

pos_counts = train_labels.sum(dim=0)
neg_counts = train_labels.shape[0] - pos_counts
pos_weight = (neg_counts / pos_counts.clamp_min(1.0)).to(device)
print("pos_weight:", pos_weight)

linear_head = nn.Linear(train_features.shape[1], NUM_CLASSES).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(linear_head.parameters(), lr=3e-3)


def run_epoch() -> float:
    linear_head.train()
    total = 0.0
    for x, y in probe_train_loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(linear_head(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / max(len(probe_train_loader), 1)


def evaluate_probe(loader):
    linear_head.eval()
    probs, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = linear_head(x.to(device))
            probs.append(torch.sigmoid(logits).cpu())
            labels.append(y.cpu())
    probs_np = torch.cat(probs).numpy()
    labels_np = torch.cat(labels).numpy()
    return {
        "macro_auroc": roc_auc_score(labels_np, probs_np, average="macro"),
        "macro_auprc": average_precision_score(labels_np, probs_np, average="macro"),
        "probs": probs_np,
        "labels": labels_np,
    }

best_state = None
best_val_auroc = -float("inf")
for epoch in range(1, 101):
    loss = run_epoch()
    val_metrics = evaluate_probe(probe_val_loader)
    if val_metrics["macro_auroc"] > best_val_auroc:
        best_val_auroc = val_metrics["macro_auroc"]
        best_state = copy.deepcopy(linear_head.state_dict())
    if epoch == 1 or epoch % 10 == 0:
        print(f"epoch {epoch:03d} | loss {loss:.4f} | val AUROC {val_metrics['macro_auroc']:.4f} | val AUPRC {val_metrics['macro_auprc']:.4f}")

linear_head.load_state_dict(best_state)
test_metrics = evaluate_probe(probe_test_loader)
preds = (test_metrics["probs"] >= 0.5).astype(int)
print(classification_report(test_metrics["labels"], preds, zero_division=0))
print(f"Best val AUROC: {best_val_auroc:.4f}")
print(f"Test Macro AUROC: {test_metrics['macro_auroc']:.4f}")
print(f"Test Macro AUPRC: {test_metrics['macro_auprc']:.4f}")

pos_weight: tensor([3.3740, 7.0674, 2.8986, 1.2494, 3.0808], device='cuda:0')
epoch 001 | loss 0.7869 | val AUROC 0.8380 | val AUPRC 0.6547
epoch 010 | loss 0.6802 | val AUROC 0.8589 | val AUPRC 0.6888
epoch 020 | loss 0.6664 | val AUROC 0.8624 | val AUPRC 0.6903
epoch 030 | loss 0.6583 | val AUROC 0.8644 | val AUPRC 0.6963
epoch 040 | loss 0.6548 | val AUROC 0.8648 | val AUPRC 0.6987
epoch 050 | loss 0.6499 | val AUROC 0.8656 | val AUPRC 0.6993
epoch 060 | loss 0.6472 | val AUROC 0.8655 | val AUPRC 0.6989
epoch 070 | loss 0.6445 | val AUROC 0.8653 | val AUPRC 0.6984
epoch 080 | loss 0.6425 | val AUROC 0.8663 | val AUPRC 0.7004
epoch 090 | loss 0.6395 | val AUROC 0.8681 | val AUPRC 0.7017
epoch 100 | loss 0.6398 | val AUROC 0.8675 | val AUPRC 0.7016
              precision    recall  f1-score   support

           0       0.54      0.68      0.60       498
           1       0.39      0.75      0.51       263
           2       0.53      0.67      0.59       553
           3       0.73